# 05. Model Evaluation and Final Selection

For a cancer classification project, accuracy alone is not enough. We must pay attention to:

- precision
- recall / sensitivity
- F1 score
- confusion matrix
- ROC-AUC
- false negatives

This project is educational and not a clinical system, but this evaluation framework mirrors real ML practice.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
data_path = project_root / 'data' / 'data.csv'

df = pd.read_csv(data_path)
df = df.drop(columns=[col for col in df.columns if 'Unnamed:' in str(col)], errors='ignore')
df = df.drop(columns=['id'], errors='ignore')
df['diagnosis'] = df['diagnosis'].astype(str).str.strip()
X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].map({'B': 0, 'M': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    'knn': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5)),
    ]),
    'svm': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
    'gradient_boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(random_state=42)),
    ]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    row = {
        'model': name,
        'accuracy': (y_pred == y_test).mean(),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
    }
    results.append(row)
    print(f'\n=== {name.upper()} ===')
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))
    print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
    print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))

best = max(results, key=lambda r: (r['roc_auc'], r['f1'], r['accuracy']))
print('\nBEST MODEL:')
print(best)

## Final takeaway

The model evaluation step decides which model is most appropriate for deployment in the project pipeline. In this dataset, the strongest candidate is not necessarily the model that achieves the highest raw accuracy alone; we care about balance across precision, recall, and ROC-AUC.

The final project should then save the best pipeline, document the results, and keep the evaluation transparent. That is the key difference between an experimental notebook and a portfolio-ready ML project.